In [ ]:
def objective_function(x):
    return -x**2 + 10*x

def hill_climbing(start, step_size=0.1):
    current = start

    while True:
        neighbors = [current - step_size, current + step_size]
        best_neighbor = max(neighbors, key=objective_function)

        if objective_function(best_neighbor) <= objective_function(current):
            break

        current = best_neighbor

    return current, objective_function(current)

# Run the algorithm
start = 0
solution, value = hill_climbing(start)

print("Best solution:", round(solution, 2))


Best solution: 5.0


In [ ]:

import math
import random

def objective_function(x):
    return -x**2 + 10*x

def simulated_annealing():
    current = random.uniform(0, 10)
    best = current
    temperature = 100
    cooling_rate = 0.95

    while temperature > 0.01:
        neighbor = current + random.uniform(-1, 1)

        delta = objective_function(neighbor) - objective_function(current)

        if delta > 0 or random.random() < math.exp(delta / temperature):
            current = neighbor

        if objective_function(current) > objective_function(best):
            best = current

        temperature *= cooling_rate

    return best

solution = simulated_annealing()

print("Best solution:", round(solution, 2))
print("Maximum value:", round(objective_function(solution), 2))


Best solution: 4.99
Maximum value: 25.0


In [ ]:

# Constraint Satisfaction Problem (CSP)
# Map Coloring using Backtracking

graph = {
    'A': ['B', 'C'],
    'B': ['A', 'C', 'D'],
    'C': ['A', 'B', 'D'],
    'D': ['B', 'C']
}

colors = ['Red', 'Green', 'Blue']
assignment = {}

def is_valid(node, color):
    for neighbor in graph[node]:
        if neighbor in assignment and assignment[neighbor] == color:
            return False
    return True

def solve_csp():
    if len(assignment) == len(graph):
        return True

    node = next(n for n in graph if n not in assignment)

    for color in colors:
        if is_valid(node, color):
            assignment[node] = color

            if solve_csp():
                return True

            del assignment[node]

    return False

if solve_csp():
    print("Solution Found:")
    for node, color in assignment.items():
        print(f"Region {node}: {color}")
else:
    print("No solution exists.")


Solution Found:
Region A: Red
Region B: Green
Region C: Blue
Region D: Red


In [4]:

import pandas as pd, random
from itertools import product
from functools import lru_cache
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Load data
data = pd.read_csv("diabetes.csv", header=None)
data.columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
X, y = data.iloc[:, :-1], data["Outcome"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=.3, random_state=42, stratify=y
)

# Population: all 256 feature combinations
population = [''.join(p) for p in product("01", repeat=8)]
features = list(X.columns)

def SelectFeatures(ch):
    return [f for f, b in zip(features, ch) if b == "1"]

@lru_cache(None)
def CalculateFitness(ch):
    cols = SelectFeatures(ch)
    if not cols:
        return 0
    model = XGBClassifier(n_estimators=100, random_state=42,
                          eval_metric="logloss", verbosity=0)
    model.fit(Xtr[cols], ytr)
    return accuracy_score(yte, model.predict(Xte[cols]))

# Evaluate each chromosome once
fitness = {ch: CalculateFitness(ch) for ch in population}

# Tournament selection: consecutive pairs
winners = [max(population[i:i+2], key=fitness.get)
           for i in range(0, len(population), 2)]

# Crossover and mutation using the first two winners
p1, p2 = winners[:2]
c1, c2 = p1[:-3] + p2[-3:], p2[:-3] + p1[-3:]

def MutateChild(ch):
    return ch[:3] + ("1" if ch[3] == "0" else "0") + ch[4:]

children = [MutateChild(c1), MutateChild(c2)]

# Compare offspring against the best original chromosome
best = max(population, key=fitness.get)
for ch in children:
    print(f"{ch} | Features: {SelectFeatures(ch)} | "
          f"Accuracy: {CalculateFitness(ch):.4f} | "
          f"{'BETTER' if fitness[ch] > fitness[best] else 'NOT BETTER'}")

# Best tournament winner
best = max(winners, key=fitness.get)
print("\nBest chromosome:", best)
print("Selected features:", SelectFeatures(best))
print("Accuracy:", round(fitness[best], 4))


FileNotFoundError: [Errno 2] No such file or directory: 'diabetes.csv'